# Jumbotail Replenishment Planner

**Suggestion generation date: 2026-03-16**

This notebook implements the take-home requirements, parses the JSON fields, accounts for on-hand inventory and open POs, rounds to cases, respects the space cap, applies MOV, and produces the required output columns.

In [ ]:
import pandas as pd, numpy as np, json, math
df = pd.read_csv('assignment_data.csv')
print('Dataset shape:', df.shape)
print('Columns:', len(df.columns))

## JSON parsing
`inventory_breakup` and `open_po_details` are parsed as JSON. The aggregate `orderedquantity` is used as the total open-PO pipeline because it is the explicit dataset-level total; the JSON detail is retained for audit/reconciliation.

In [ ]:
def parse_json(s):
    if pd.isna(s) or not str(s).strip(): return {}
    try:
        x=json.loads(s)
        return x if isinstance(x,dict) else {}
    except Exception:
        return {}

def po_items(s): return list(parse_json(s).values())

df["_parsed_po_units"]=df["open_po_details"].apply(
    lambda s: sum(float(x.get("orderedquantity") or 0) for x in po_items(s))
)
df["_open_po_units"]=pd.to_numeric(df["orderedquantity"],errors="coerce").fillna(0)

print("Rows:",len(df))
print("PO-detail JSON rows:",int(df.open_po_details.notna().sum()))
print("Aggregate open-PO units:",int(df["_open_po_units"].sum()))

## Replenishment logic
Target DOI = `inv_norm + safety_stock`.

Required units = max(target units − on-hand − open PO units, 0).

Required units are rounded up to whole cases. The order is capped at the largest whole-case quantity that does not exceed `max_allocated_space`. If MOV cannot be achieved within that cap, the planner keeps the maximum feasible order and explicitly flags `MOV_NOT_MET_SPACE_LIMIT`.

In [ ]:
from replenishment_planner import build_planner
result = build_planner(pd.read_csv('assignment_data.csv'))
result.head()

## Constraint validation

In [ ]:
print("Whole-case:", (result.final_suggestion == result.final_cases_suggestion*result.case_size).all())
print("Space cap:", (result.final_suggestion <= result.max_allocated_space).all())
print("Non-negative:", (result.final_suggestion >= 0).all())
print("Suggested units:", int(result.final_suggestion.sum()))
print("Suggested cases:", int(result.final_cases_suggestion.sum()))
print("Suggested value:", round(result.final_value.sum(),2))
print("MOV exceptions:", int((result.mov_check=="MOV_NOT_MET_SPACE_LIMIT").sum()))

## SQL
See `replenishment_queries.sql` for the required vendor-level dashboard query, top-10 risk query, and a bonus sales-band inventory-health query.